# 08 目标检测与语义分割

## 超越分类：定位与分割

图像分类只能回答"图片里有什么"，但实际应用往往需要更多：

### 目标检测（Object Detection）
- **回答的问题**：图片中**有哪些物体**？每个物体在**什么位置**？
- **输出**：边界框（Bounding Box）+ 类别标签 + 置信度
- **代表算法**：YOLO系列、Faster R-CNN、SSD

### 语义分割（Semantic Segmentation）
- **回答的问题**：图片中**每个像素**属于什么类别？
- **输出**：像素级别的类别标注
- **代表算法**：DeepLab系列、U-Net、SegFormer

### 两者的关系

| 任务 | 输出粒度 | 典型应用 |
|------|----------|----------|
| 图像分类 | 全图-1个标签 | 判断图片是猫还是狗 |
| 目标检测 | 物体级别-多个框 | 自动驾驶检测行人/车辆 |
| 语义分割 | 像素级别-密集预测 | 医学图像器官分割 |

本教程将使用 YOLOv8 进行目标检测，使用 DeepLabV3 进行语义分割，并比较两种方法。

In [ ]:
# ===== 环境准备 =====
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
import warnings
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import torchvision
from torchvision import transforms

import cv2
from PIL import Image
from pathlib import Path

# 字体设置
mpl.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'DejaVu Sans']
mpl.rcParams['axes.unicode_minus'] = False

# 设备
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"PyTorch {torch.__version__}  |  torchvision {torchvision.__version__}")
print(f"OpenCV {cv2.__version__}")
print(f"计算设备: {DEVICE}")
if torch.cuda.is_available():
    print(f"  GPU: {torch.cuda.get_device_name(0)}")


## YOLOv8 目标检测

YOLO（You Only Look Once）是最流行的实时目标检测算法之一。YOLOv8是Ultralytics公司的最新版本，具有速度快、精度高的特点。

我们将使用YOLOv8的nano版本（yolov8n.pt），它是最轻量的变体，适合快速实验。

In [ ]:
# ===== YOLOv8 安装与加载 =====

# 检查并安装ultralytics
try:
    from ultralytics import YOLO
    print("ultralytics 已安装")
except ImportError:
    print("正在安装 ultralytics...")
    import subprocess, sys
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'ultralytics', '-q'])
    from ultralytics import YOLO
    print("ultralytics 安装完成")

# 加载YOLOv8n预训练模型
model_yolo = YOLO('yolov8n.pt')
print(f"YOLOv8n 模型加载成功")

# 准备一张示例图片
# 使用skimage的示例图片或创建一张
try:
    from skimage import data
    sample_img = data.astronaut()  # 宇航员照片 512x512 RGB
    print(f"示例图片: astronaut ({sample_img.shape[1]}x{sample_img.shape[0]})")
except ImportError:
    # 创建一张简单的测试图片
    sample_img = np.ones((512, 512, 3), dtype=np.uint8) * 200
    cv2.rectangle(sample_img, (100, 100), (300, 300), (0, 0, 255), -1)
    cv2.circle(sample_img, (400, 300), 80, (255, 0, 0), -1)
    cv2.rectangle(sample_img, (50, 350), (200, 450), (0, 255, 0), -1)
    print(f"示例图片: 合成图片 ({sample_img.shape[1]}x{sample_img.shape[0]})")

# 显示原始图片
plt.figure(figsize=(8, 8))
plt.imshow(sample_img)
plt.title('原始图片', fontsize=14)
plt.axis('off')
plt.show()

# YOLOv8推理
results = model_yolo(sample_img, verbose=False)
print(f"\nYOLOv8 检测完成")
print(f"检测到的物体数量: {len(results[0].boxes)}")


## 解析检测结果

YOLO的输出包含每个检测框的信息：边界框坐标（xyxy格式）、类别ID和置信度分数。我们来详细解析这些结果。

In [ ]:
# ===== 解析YOLO检测结果 =====

result = results[0]
boxes = result.boxes  # Boxes对象

if len(boxes) > 0:
    # 提取检测信息
    xyxy = boxes.xyxy.cpu().numpy()       # [x1, y1, x2, y2] 坐标
    confidences = boxes.conf.cpu().numpy() # 置信度
    class_ids = boxes.cls.cpu().numpy().astype(int)  # 类别ID

    print(f"检测到 {len(xyxy)} 个物体:\n")
    print(f"{'索引':<6} {'类别':<15} {'类别ID':<8} {'置信度':<10} {'边界框 (x1,y1,x2,y2)'}")
    print("-" * 75)

    for i in range(len(xyxy)):
        class_name = model_yolo.names[class_ids[i]]
        x1, y1, x2, y2 = xyxy[i]
        print(f"{i:<6} {class_name:<15} {class_ids[i]:<8} "
              f"{confidences[i]:<10.4f} [{x1:.0f}, {y1:.0f}, {x2:.0f}, {y2:.0f}]")

    # 按置信度排序
    sorted_idx = np.argsort(confidences)[::-1]
    print(f"\n置信度最高的检测:")
    top_idx = sorted_idx[0]
    print(f"  {model_yolo.names[class_ids[top_idx]]} "
          f"(置信度: {confidences[top_idx]:.4f})")
else:
    print("未检测到任何物体。尝试使用包含物体的图片。")
    xyxy = np.array([])
    confidences = np.array([])
    class_ids = np.array([])


## 绘制检测边界框

检测到目标后，我们需要在图片上绘制边界框和标签。使用OpenCV的`rectangle`和`putText`函数，为每个检测到的物体绘制彩色框和标签。

In [ ]:
# ===== 绘制检测边界框 =====

def draw_detections(image, boxes_xyxy, class_ids, confidences,
                    class_names, score_threshold=0.3):
    '''
    在图像上绘制目标检测结果。

    参数:
        image: 原始图片 (numpy array, RGB)
        boxes_xyxy: 边界框 [[x1, y1, x2, y2], ...]
        class_ids: 类别ID列表
        confidences: 置信度列表
        class_names: 类别名称字典 {id: name}
        score_threshold: 置信度阈值

    返回:
        result_img: 绘制后的图片
    '''
    result_img = image.copy()

    # 为不同类别生成不同颜色
    np.random.seed(42)
    colors = {}
    for cid in np.unique(class_ids):
        colors[cid] = tuple(np.random.randint(50, 255, 3).tolist())

    for i in range(len(boxes_xyxy)):
        if confidences[i] < score_threshold:
            continue

        x1, y1, x2, y2 = boxes_xyxy[i].astype(int)
        cid = class_ids[i]
        conf = confidences[i]
        name = class_names.get(cid, f'Class {cid}')

        # 确保RGB到BGR的转换（cv2使用BGR）
        if result_img.shape[2] == 3:
            img_bgr = cv2.cvtColor(result_img, cv2.COLOR_RGB2BGR)
        else:
            img_bgr = result_img

        color = colors.get(cid, (0, 255, 0))

        # 绘制边界框
        cv2.rectangle(img_bgr, (x1, y1), (x2, y2), color, 2)

        # 绘制标签背景
        label = f"{name} {conf:.2f}"
        (label_w, label_h), baseline = cv2.getTextSize(
            label, cv2.FONT_HERSHEY_SIMPLEX, 0.5, 2
        )
        cv2.rectangle(img_bgr, (x1, y1 - label_h - 10),
                      (x1 + label_w, y1), color, -1)

        # 绘制标签文字
        cv2.putText(img_bgr, label, (x1, y1 - 5),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 2)

        # 转回RGB
        result_img = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

    return result_img


# 应用绘制函数
if len(xyxy) > 0:
    detected_img = draw_detections(
        sample_img, xyxy, class_ids, confidences,
        model_yolo.names, score_threshold=0.3
    )
else:
    detected_img = sample_img.copy()

# 显示结果
plt.figure(figsize=(10, 10))
plt.imshow(detected_img)
plt.title(f'YOLOv8n 检测结果 ({len(xyxy)} 个物体)', fontsize=14)
plt.axis('off')
plt.show()


## 非极大值抑制（NMS）

YOLO等目标检测算法通常会为同一个物体产生多个重叠的检测框。**非极大值抑制（Non-Maximum Suppression, NMS）**用于去除冗余的检测框，保留最佳的检测结果。

NMS的工作流程：
1. 按置信度降序排列所有检测框
2. 选择置信度最高的框，将其加入最终结果
3. 计算该框与其余所有框的IoU，删除IoU超过阈值的框
4. 重复步骤2-3，直到所有框都被处理

In [ ]:
# ===== IoU计算 =====

def compute_iou(box1, box2):
    '''
    计算两个边界框的IoU (Intersection over Union)。

    参数:
        box1, box2: [x1, y1, x2, y2]

    返回:
        iou: 交并比 [0, 1]
    '''
    # 计算交集
    x1_inter = max(box1[0], box2[0])
    y1_inter = max(box1[1], box2[1])
    x2_inter = min(box1[2], box2[2])
    y2_inter = min(box1[3], box2[3])

    # 如果没有重叠
    if x2_inter <= x1_inter or y2_inter <= y1_inter:
        return 0.0

    inter_area = (x2_inter - x1_inter) * (y2_inter - y1_inter)

    # 计算各自的面积
    area1 = (box1[2] - box1[0]) * (box1[3] - box1[1])
    area2 = (box2[2] - box2[0]) * (box2[3] - box2[1])

    # 计算并集面积
    union_area = area1 + area2 - inter_area

    iou = inter_area / union_area if union_area > 0 else 0.0
    return iou


def nms(boxes, scores, iou_threshold=0.5):
    '''
    非极大值抑制 (NMS)。

    参数:
        boxes: 边界框列表 [[x1, y1, x2, y2], ...]
        scores: 置信度分数列表
        iou_threshold: IoU阈值，超过此阈值的框会被抑制

    返回:
        keep_indices: 保留的框的索引
    '''
    if len(boxes) == 0:
        return []

    boxes = np.array(boxes)
    scores = np.array(scores)

    # 按分数降序排序
    order = scores.argsort()[::-1]

    keep = []
    suppressed = set()

    while len(order) > 0:
        idx = order[0]
        keep.append(idx)
        order = order[1:]

        new_order = []
        for j in order:
            iou = compute_iou(boxes[idx], boxes[j])
            if iou <= iou_threshold:
                new_order.append(j)

        order = np.array(new_order)

    return keep


# ---- 演示：NMS前后对比 ----
# 创建一些模拟的重叠检测框
np.random.seed(42)
demo_boxes = np.array([
    [100, 100, 300, 300],  # 主要框
    [110, 90, 290, 310],   # 高度重叠
    [120, 110, 280, 290],  # 高度重叠
    [95, 105, 305, 295],   # 高度重叠
    [400, 200, 500, 400],  # 另一个物体
    [380, 180, 520, 420],  # 部分重叠
    [50, 350, 180, 500],   # 第三个物体
], dtype=np.float64)

demo_scores = np.array([0.95, 0.85, 0.75, 0.65, 0.90, 0.70, 0.80])

# 执行NMS
keep_indices = nms(demo_boxes, demo_scores, iou_threshold=0.5)

# 可视化对比
fig, axes = plt.subplots(1, 2, figsize=(16, 8))

# 创建演示画布
canvas = np.ones((600, 600, 3), dtype=np.uint8) * 255

# NMS前
canvas_before = canvas.copy()
for i, (box, score) in enumerate(zip(demo_boxes, demo_scores)):
    x1, y1, x2, y2 = box.astype(int)
    cv2.rectangle(canvas_before, (x1, y1), (x2, y2), (0, 150, 0), 2)
    cv2.putText(canvas_before, f'#{i}({score:.2f})', (x1, y1-5),
                cv2.FONT_HERSHEY_SIMPLEX, 0.4, (0, 100, 0), 1)

axes[0].imshow(canvas_before)
axes[0].set_title(f'NMS前 ({len(demo_boxes)} 个框)', fontsize=12)
axes[0].axis('off')

# NMS后
canvas_after = canvas.copy()
colors_after = [(0, 0, 255), (255, 0, 0), (0, 255, 0)]
for rank, idx in enumerate(keep_indices):
    box = demo_boxes[idx]
    score = demo_scores[idx]
    x1, y1, x2, y2 = box.astype(int)
    color = colors_after[rank % len(colors_after)]
    cv2.rectangle(canvas_after, (x1, y1), (x2, y2), color, 3)
    cv2.putText(canvas_after, f'#{idx}({score:.2f})', (x1, y1-5),
                cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)

axes[1].imshow(canvas_after)
axes[1].set_title(f'NMS后 ({len(keep_indices)} 个框, IoU阈值=0.5)', fontsize=12)
axes[1].axis('off')

plt.suptitle('非极大值抑制 (NMS) 演示', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print(f"NMS 结果: {len(demo_boxes)} 个候选框 → {len(keep_indices)} 个保留框")
print(f"保留的框索引: {keep_indices}")


## IoU可视化

IoU（Intersection over Union，交并比）是目标检测中最基础的评估指标。它衡量两个边界框的重叠程度，取值范围为[0, 1]，1表示完全重合。

In [ ]:
# ===== IoU可视化 =====

def visualize_iou(box1, box2):
    '''可视化两个边界框及其IoU'''
    iou = compute_iou(box1, box2)

    fig, ax = plt.subplots(figsize=(10, 8))

    canvas = np.ones((400, 400, 3), dtype=np.uint8) * 255

    # 绘制box1（红色）
    x1, y1, x2, y2 = box1.astype(int)
    cv2.rectangle(canvas, (x1, y1), (x2, y2), (255, 0, 0), 2)
    cv2.putText(canvas, 'Box A', (x1, y1-10),
                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 0, 0), 2)

    # 绘制box2（蓝色）
    x1, y1, x2, y2 = box2.astype(int)
    cv2.rectangle(canvas, (x1, y1), (x2, y2), (0, 0, 255), 2)
    cv2.putText(canvas, 'Box B', (x1, y1-10),
                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 255), 2)

    # 计算并绘制交集区域
    x1_inter = max(box1[0], box2[0])
    y1_inter = max(box1[1], box2[1])
    x2_inter = min(box1[2], box2[2])
    y2_inter = min(box1[3], box2[3])

    if x2_inter > x1_inter and y2_inter > y1_inter:
        inter_box = np.array([x1_inter, y1_inter, x2_inter, y2_inter]).astype(int)
        cv2.rectangle(canvas, (inter_box[0], inter_box[1]),
                      (inter_box[2], inter_box[3]), (0, 255, 0), -1)
        cv2.putText(canvas, 'Intersection', (inter_box[0], inter_box[1]-10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 150, 0), 1)

    ax.imshow(canvas)
    ax.set_title(f'IoU = {iou:.4f}', fontsize=14)
    ax.axis('off')
    plt.show()

    return iou


# 测试不同的IoU场景
print("=== IoU 计算与可视化 ===\n")

# 场景1：高度重叠
box_a = np.array([100, 100, 300, 300])
box_b = np.array([150, 120, 320, 280])
iou1 = visualize_iou(box_a, box_b)
print(f"场景1 - 高度重叠: IoU = {iou1:.4f}")

# 场景2：部分重叠
box_a2 = np.array([100, 100, 250, 250])
box_b2 = np.array([180, 180, 350, 350])
iou2 = visualize_iou(box_a2, box_b2)
print(f"场景2 - 部分重叠: IoU = {iou2:.4f}")

# 场景3：几乎无重叠
box_a3 = np.array([50, 50, 150, 150])
box_b3 = np.array([250, 250, 350, 350])
iou3 = visualize_iou(box_a3, box_b3)
print(f"场景3 - 无重叠: IoU = {iou3:.4f}")


## 语义分割：DeepLabV3

DeepLabV3是Google提出的语义分割模型，使用**空洞卷积（Atrous Convolution）**和**ASPP（Atrous Spatial Pyramid Pooling）**来捕获多尺度的上下文信息。

我们将使用PyTorch预训练的DeepLabV3 with ResNet-101骨干网络，它在PASCAL VOC数据集上训练，可以将图像分割为21个类别（包括背景、人、汽车、狗、猫等）。

In [ ]:
# ===== 语义分割：加载DeepLabV3 =====

# 加载预训练模型
model_deeplab = torchvision.models.segmentation.deeplabv3_resnet101(
    weights=torchvision.models.segmentation.DeepLabV3_ResNet101_Weights.COCO_WITH_VOC_LABELS_V1
)
model_deeplab = model_deeplab.to(DEVICE)
model_deeplab.eval()

print("DeepLabV3 (ResNet-101) 加载成功")
print(f"类别数: 21 (PASCAL VOC)")
total_params = sum(p.numel() for p in model_deeplab.parameters())
print(f"总参数量: {total_params:,}")

# 在示例图片上运行分割
preprocess = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

input_tensor = preprocess(sample_img).unsqueeze(0).to(DEVICE)

with torch.no_grad():
    output = model_deeplab(input_tensor)['out']
    # output shape: [1, 21, H, W]
    print(f"\n分割输出形状: {list(output.shape)}")
    print(f"  批次: 1, 通道(类别): 21, 高度: {output.shape[2]}, 宽度: {output.shape[3]}")

# 获取每个像素的预测类别
seg_pred = output.argmax(dim=1).squeeze(0).cpu().numpy()
print(f"分割结果形状: {seg_pred.shape}")
print(f"预测的类别: {np.unique(seg_pred)}")


In [ ]:
# ===== 处理DeepLabV3输出 =====

# PASCAL VOC 21类别的颜色映射
VOC_COLORMAP = {
    0:  [0, 0, 0],        # 背景
    1:  [128, 0, 0],      # 飞机
    2:  [0, 128, 0],      # 自行车
    3:  [128, 128, 0],    # 鸟
    4:  [0, 0, 128],      # 船
    5:  [128, 0, 128],    # 瓶子
    6:  [0, 128, 128],    # 公交车
    7:  [128, 128, 128],  # 汽车
    8:  [64, 0, 0],       # 猫
    9:  [192, 0, 0],      # 椅子
    10: [64, 128, 0],     # 牛
    11: [192, 128, 0],    # 餐桌
    12: [64, 0, 128],     # 狗
    13: [192, 0, 128],    # 马
    14: [64, 128, 128],   # 摩托车
    15: [192, 128, 128],  # 人
    16: [0, 64, 0],       # 盆栽
    17: [128, 64, 0],     # 羊
    18: [0, 192, 0],      # 沙发
    19: [128, 192, 0],    # 火车
    20: [0, 64, 128],     # 电视/显示器
}

VOC_CLASSES = ['背景', '飞机', '自行车', '鸟', '船', '瓶子', '公交车',
               '汽车', '猫', '椅子', '牛', '餐桌', '狗', '马', '摩托车',
               '人', '盆栽', '羊', '沙发', '火车', '电视']

def create_segmentation_overlay(image, seg_map, alpha=0.6):
    '''将分割结果叠加到原图上'''
    # 创建彩色分割图
    H, W = seg_map.shape
    colored_mask = np.zeros((H, W, 3), dtype=np.uint8)

    for class_id, color in VOC_COLORMAP.items():
        colored_mask[seg_map == class_id] = color

    # 调整尺寸匹配原图
    if colored_mask.shape[:2] != image.shape[:2]:
        colored_mask = cv2.resize(colored_mask, (image.shape[1], image.shape[0]),
                                  interpolation=cv2.INTER_NEAREST)

    # 叠加
    overlay = cv2.addWeighted(image, 1 - alpha, colored_mask, alpha, 0)

    return overlay, colored_mask


# 调整分割结果到原图尺寸
seg_map_resized = cv2.resize(seg_pred, (sample_img.shape[1], sample_img.shape[0]),
                              interpolation=cv2.INTER_NEAREST)

# 创建叠加图
overlay_img, colored_mask = create_segmentation_overlay(sample_img, seg_map_resized, alpha=0.5)

# 可视化
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

axes[0].imshow(sample_img)
axes[0].set_title('原始图片', fontsize=12)
axes[0].axis('off')

axes[1].imshow(colored_mask)
axes[1].set_title('语义分割掩膜', fontsize=12)
axes[1].axis('off')

axes[2].imshow(overlay_img)
axes[2].set_title('叠加效果 (alpha=0.5)', fontsize=12)
axes[2].axis('off')

plt.suptitle('DeepLabV3 语义分割结果', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

# 统计各类别像素占比
unique_classes, counts = np.unique(seg_map_resized, return_counts=True)
total_pixels = seg_map_resized.size
print("\n各类别像素占比:")
for cls_id, count in zip(unique_classes, counts):
    if cls_id < len(VOC_CLASSES):
        percentage = 100 * count / total_pixels
        if percentage > 1.0:  # 只显示占比>1%的类别
            print(f"  {VOC_CLASSES[cls_id]:<8} {percentage:5.1f}%")


## 目标检测 vs 语义分割对比

将YOLO的目标检测结果和DeepLabV3的语义分割结果并排显示，直观对比两种方法：
- 目标检测输出的是**稀疏的边界框**，告诉你物体的位置和类别
- 语义分割输出的是**密集的像素级分类**，精确到每个像素

In [ ]:
# ===== 检测 vs 分割 对比 =====

# 确保有检测结果
if len(xyxy) == 0:
    # 如果没有检测结果，使用模拟数据
    xyxy = np.array([[100, 80, 390, 450], [200, 250, 350, 380]])
    class_ids = np.array([15, 8])  # 人, 猫
    confidences = np.array([0.95, 0.88])

# 生成检测图
detection_img = draw_detections(
    sample_img, xyxy, class_ids, confidences,
    model_yolo.names, score_threshold=0.3
)

# 生成分割图
_, seg_colored = create_segmentation_overlay(sample_img, seg_map_resized, alpha=0.6)

# 对比展示
fig, axes = plt.subplots(1, 3, figsize=(20, 7))

axes[0].imshow(sample_img)
axes[0].set_title('原始图片', fontsize=13)
axes[0].axis('off')

axes[1].imshow(detection_img)
axes[1].set_title(f'YOLOv8 目标检测\n(稀疏框: {len(xyxy)} 个)', fontsize=13)
axes[1].axis('off')

axes[2].imshow(seg_colored)
axes[2].set_title('DeepLabV3 语义分割\n(密集预测: 像素级)', fontsize=13)
axes[2].axis('off')

plt.suptitle('目标检测 vs 语义分割', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print("目标检测 vs 语义分割:")
print("  YOLO检测: 输出边界框（物体位置 + 类别）")
print("  DeepLabV3: 输出像素级分类（每个像素的类别）")
print("\n实际应用中两者常结合使用:")
print("  检测定位个体 + 分割精确描绘轮廓")
print("  例如自动驾驶: 检测车辆位置 + 分割道路区域")


## 练习与实践

In [ ]:
# TODO: 练习1 - 调整YOLO的置信度和IoU阈值
# 观察不同阈值对检测结果的影响

print("练习1: YOLO阈值调整实验")
print("=" * 50)

# 尝试不同的置信度阈值
conf_thresholds = [0.1, 0.25, 0.5, 0.75]
iou_thresholds = [0.3, 0.5, 0.7]

print("参数组合:")
for ct in conf_thresholds:
    for iot in iou_thresholds:
        # 使用不同参数运行YOLO
        results_test = model_yolo(sample_img, conf=ct, iou=iot, verbose=False)
        n_boxes = len(results_test[0].boxes)
        print(f"  conf={ct:.2f}, iou={iot:.1f} → {n_boxes} 个检测框")

print("\n问题思考:")
print("  1. 置信度阈值过低会导致什么？")
print("  2. IoU阈值过高会导致什么？")
print("  3. 如何选择适合你应用场景的阈值？")
print("  (提示: 安防场景需要高召回率，推荐系统需要高精确率)")

# TODO: 请尝试修改上述参数，观察检测结果的变化
# 并思考在不同应用场景下应该如何设置阈值


In [ ]:
# TODO: 练习2 - 使用DeepLabV3的MobileNet版本
# 比较与ResNet101版本的速度和分割质量

print("练习2: DeepLabV3 骨干网络对比")
print("=" * 50)

# 加载MobileNetV3版本
try:
    model_mobilenet = torchvision.models.segmentation.deeplabv3_mobilenet_v3_large(
        weights=torchvision.models.segmentation.DeepLabV3_MobileNet_V3_Large_Weights.COCO_WITH_VOC_LABELS_V1
    )
    model_mobilenet = model_mobilenet.to(DEVICE)
    model_mobilenet.eval()

    # 比较参数量
    params_resnet = sum(p.numel() for p in model_deeplab.parameters())
    params_mobilenet = sum(p.numel() for p in model_mobilenet.parameters())

    print(f"ResNet101版本: {params_resnet:,} 参数")
    print(f"MobileNetV3版本: {params_mobilenet:,} 参数")
    print(f"参数比: {params_resnet/params_mobilenet:.1f}x")

    # 比较推理速度
    import time

    # ResNet101速度
    times_resnet = []
    for _ in range(10):
        start = time.time()
        with torch.no_grad():
            _ = model_deeplab(input_tensor)['out']
        times_resnet.append(time.time() - start)

    # MobileNet速度
    times_mobile = []
    for _ in range(10):
        start = time.time()
        with torch.no_grad():
            _ = model_mobilenet(input_tensor)['out']
        times_mobile.append(time.time() - start)

    print(f"\n推理速度对比 (10次平均):")
    print(f"  ResNet101:   {np.mean(times_resnet)*1000:.1f} ms (std={np.std(times_resnet)*1000:.1f})")
    print(f"  MobileNetV3: {np.mean(times_mobile)*1000:.1f} ms (std={np.std(times_mobile)*1000:.1f})")
    print(f"  加速比: {np.mean(times_resnet)/np.mean(times_mobile):.2f}x")

    # TODO: 比较两个模型在同一图片上的分割质量
    # 观察MobileNet版本是否在边缘细节上有所损失

except Exception as e:
    print(f"MobileNetV3版本加载失败: {e}")
    print("提示: 可能需要更新torchvision版本")


## 实战案例：自动驾驶场景感知模拟

自动驾驶系统需要同时感知周围环境中的多种信息。本案例演示目标检测和语义分割如何协同工作：

- **YOLO** 检测动态物体（汽车、行人、交通标志）——位置和类别
- **DeepLabV3** 分割静态场景（道路、天空、植被、建筑）——可行驶区域
- **两者结合** 提供完整的场景理解

In [ ]:
# ===== 实战：自动驾驶场景感知模拟 =====

print("=" * 60)
print("自动驾驶场景感知模拟")
print("=" * 60)

# 创建模拟的街景图片
street_img = np.ones((400, 600, 3), dtype=np.uint8)

# 天空（顶部）
street_img[:180, :] = [135, 206, 235]  # 淡蓝

# 建筑
street_img[80:250, :80] = [180, 170, 160]
street_img[60:220, 500:] = [160, 155, 145]

# 道路
street_img[250:, :] = [80, 80, 80]  # 深灰

# 车道线
for x in range(0, 600, 60):
    street_img[300:308, x:x+30] = [255, 255, 255]

# 车辆
cv2.rectangle(street_img, (150, 260), (220, 310), (0, 0, 255), -1)  # 红车
cv2.rectangle(street_img, (350, 270), (430, 330), (255, 0, 0), -1)  # 蓝车

# 行人
cv2.circle(street_img, (480, 300), 15, (0, 200, 0), -1)
cv2.rectangle(street_img, (470, 315), (490, 360), (0, 200, 0), -1)

# 交通灯
cv2.circle(street_img, (100, 120), 12, (0, 255, 0), -1)  # 绿灯

# 树木
cv2.circle(street_img, (50, 230), 40, (34, 139, 34), -1)
cv2.circle(street_img, (560, 220), 45, (34, 139, 34), -1)

plt.figure(figsize=(8, 6))
plt.imshow(street_img)
plt.title('模拟街景', fontsize=14)
plt.axis('off')
plt.show()

# ---- YOLO检测 ----
street_results = model_yolo(street_img, verbose=False, conf=0.3)
street_boxes = street_results[0].boxes

if len(street_boxes) > 0:
    street_xyxy = street_boxes.xyxy.cpu().numpy()
    street_cls = street_boxes.cls.cpu().numpy().astype(int)
    street_conf = street_boxes.conf.cpu().numpy()

    detect_img = draw_detections(street_img, street_xyxy, street_cls,
                                 street_conf, model_yolo.names, 0.3)
else:
    # 手动标注模拟
    detect_img = street_img.copy()
    cv2.rectangle(detect_img, (140, 250), (230, 320), (0, 255, 0), 2)
    cv2.putText(detect_img, 'car', (140, 245), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)
    cv2.rectangle(detect_img, (340, 260), (440, 340), (0, 255, 0), 2)
    cv2.putText(detect_img, 'car', (340, 255), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)
    cv2.rectangle(detect_img, (465, 290), (500, 365), (0, 255, 255), 2)
    cv2.putText(detect_img, 'person', (465, 285), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 255), 2)

# ---- DeepLabV3分割 ----
street_tensor = preprocess(street_img).unsqueeze(0).to(DEVICE)
with torch.no_grad():
    street_seg = model_deeplab(street_tensor)['out']
    street_seg_pred = street_seg.argmax(dim=1).squeeze(0).cpu().numpy()

street_seg_resized = cv2.resize(street_seg_pred, (600, 400), interpolation=cv2.INTER_NEAREST)
_, seg_overlay = create_segmentation_overlay(street_img, street_seg_resized, alpha=0.5)

# ---- 综合展示 ----
fig, axes = plt.subplots(1, 3, figsize=(20, 7))

axes[0].imshow(street_img)
axes[0].set_title('原始街景', fontsize=13)
axes[0].set_xlabel('模拟场景: 天空+建筑+道路+车辆+行人+交通灯', fontsize=10)
axes[0].axis('off')

axes[1].imshow(detect_img)
axes[1].set_title('YOLOv8 目标检测\n检测动态物体（车辆+行人）', fontsize=13)
axes[1].axis('off')

axes[2].imshow(seg_overlay)
axes[2].set_title('DeepLabV3 语义分割\n分割静态场景（道路+天空+建筑+植被）', fontsize=13)
axes[2].axis('off')

plt.suptitle('自动驾驶场景感知 — 检测 + 分割 协同工作', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print("\n场景感知分析:")
print("  YOLO检测 (动态物体):")
print("    - 车辆位置 → 用于避障和路径规划")
print("    - 行人位置 → 安全距离监控")
print("    - 交通灯状态 → 行为决策")
print("\n  DeepLabV3分割 (静态场景):")
print("    - 道路区域 → 可行驶区域确定")
print("    - 天空区域 → 摄像头曝光控制")
print("    - 植被区域 → 场景分类")
print("\n  两者协同 → 完整的自动驾驶感知系统")
print("    检测回答'有什么，在哪里'")
print("    分割回答'这是什么区域'")
print("    结合 → 安全决策的基础")
